# MAHIR - Google Colab'da uzak GPU ile PaddleOCR

Bu not defteri, [MAHIR-PROTOTIP-HTML](https://github.com/peykan1071/MAHIR-PROTOTIP-HTML) reposunu Colab'a klonlar, yalnızca OCR işini yapan `backend/run_ocr_worker.py` sunucusunu Colab'ın ücretsiz GPU'suyla çalıştırır ve `cloudflared` ile dışarıya (kendi bilgisayarınıza) açık bir adres verir. Yerel makinenizdeki asıl MAHIR uygulaması (`backend/run_file_receiver.py`) bu adrese görsel gönderip sonucu geri alır; PaddleOCR yalnızca burada, Colab'da çalışır.

**Kullanım sırası:**
1. Üstteki menüden **Çalışma zamanı > Çalışma zamanı türünü değiştir > T4 GPU** seçin.
2. Aşağıdaki tüm hücreleri sırayla çalıştırın.
3. Son hücrede basılan `https://xxxx.trycloudflare.com` adresini kopyalayın.
4. Kendi bilgisayarınızda `README.md`'deki "Google Colab ile OCR" bölümündeki adımlarla bu adresi kullanarak yerel sunucuyu başlatın.

**Not:** Colab oturumu boşta kalınca veya ~12 saat sonra kapanır; kapanırsa bu not defterini yeniden çalıştırıp yeni adresi yerel makinenizde güncellemeniz gerekir.

In [ ]:
import os

REPO_URL = "https://github.com/peykan1071/MAHIR-PROTOTIP-HTML.git"
REPO_DIR = "/content/MAHIR-PROTOTIP-HTML"

if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

%cd {REPO_DIR}

In [ ]:
# Python sürümünü ve GPU'yu doğrula. paddleocr==3.7.0 / paddlepaddle-gpu==3.3.1 belirli
# Python sürümleri için tekerlek (wheel) yayımlıyor - burada basılan sürüm bir sonraki
# hücredeki pip install ile uyuşmazsa (ör. "tekerlek bulunamadı" hatası), cu126 indeksinin
# bu sürüm için bir tekerleği olup olmadığı
# https://www.paddlepaddle.org.cn/packages/stable/cu126/ adresinden kontrol edilmeli.
!python --version
!nvidia-smi

In [ ]:
# Yalnızca Colab tarafının ihtiyaç duyduğu OCR bağımlılıkları - yerel makinede bunlara
# hiç gerek yok, PaddleOCR sadece burada çalışıyor.
!pip install -q paddleocr==3.7.0 paddlex==3.7.2
!pip install -q paddlepaddle-gpu==3.3.1 -i https://www.paddlepaddle.org.cn/packages/stable/cu126/

In [ ]:
import os
import re
import subprocess
import time

if not os.path.exists("./cloudflared"):
    !wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
    !chmod +x cloudflared

server_process = subprocess.Popen(
    ["python", "backend/run_ocr_worker.py"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)
time.sleep(5)  # sunucunun modelleri yükleyip başlaması için kısa bir bekleme

tunnel_process = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://localhost:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

public_url = None
for _ in range(60):
    line = tunnel_process.stdout.readline()
    if not line:
        continue
    match = re.search(r"https://[a-zA-Z0-9\-]+\.trycloudflare\.com", line)
    if match:
        public_url = match.group(0)
        break

if public_url:
    print("=" * 60)
    print("YEREL MAKINENIZDE KULLANILACAK ADRES:")
    print(public_url)
    print("=" * 60)
else:
    print("Tünel adresi bulunamadı, cloudflared çıktısını kontrol edin.")

## Sunucu loglarını izlemek isterseniz

Yukarıdaki hücre çalışırken sunucu (`server_process`) arka planda çalışmaya devam eder. OCR isteklerini/hataları görmek için aşağıdaki hücreyi istediğiniz zaman tekrar çalıştırabilirsiniz.

In [ ]:
print(server_process.stdout.readline())